#### Dados de Lojas Físicas

Leitura dos dados de lojas físicas e produtos perecíveis do Squad 3.

#### 1. Configuração

Definição das informações necessárias para autenticação e acesso ao Azure Data Lake.

In [0]:
storage_account_name = "internshipdatalake"

config = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": "<CLIENT_ID>",
    "fs.azure.account.oauth2.client.secret": "<CLIENT_SECRET>",
    "fs.azure.account.oauth2.client.endpoint":
        "https://login.microsoftonline.com/<TENANT_ID>/oauth2/token"
}

#### 2. Conexão com o Data Lake

Conexão com o Azure Data Lake para validação do acesso ao ambiente de armazenamento.

In [0]:
caminho_squad = "abfss://squad3@internshipdatalake.dfs.core.windows.net/"

df_arquivos = (
    spark.read
        .format("binaryFile")
        .options(**config)
        .option("recursiveFileLookup", "true")
        .load(caminho_squad)
)

display(
    df_arquivos.select(
        "path",
        "length",
        "modificationTime"
    )
)

In [0]:
%pip install azure-storage-file-datalake azure-identity

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(
    tenant_id="<TENANT_ID>",
    client_id="<CLIENT_ID>",
    client_secret="<CLIENT_SECRET>"
)

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)

print("Cliente do Data Lake criado com sucesso!")

#### 3. Validação dos containers

Listagem dos containers disponíveis para confirmar o acesso e identificar a estrutura do Data Lake.

In [0]:
file_systems = service_client.list_file_systems()

for fs in file_systems:
    print(fs["name"])

#### 4. Exploração dos arquivos

Listagem dos arquivos disponíveis no container raw para identificação dos dados atribuídos ao Squad 3.

In [0]:
file_system_client = service_client.get_file_system_client(
    file_system="raw"
)

paths = file_system_client.get_paths()

for path in paths:
    print(path.name)

#### 5. Seleção das tabelas

Seleção das tabelas de lojas físicas e produtos perecíveis que serão utilizadas na análise.

In [0]:
tabelas = [
    "physical_lojas",
    "physical_produtos_pereciveis"
]

#### 6. Leitura das tabelas

Leitura dos arquivos CSV utilizando PySpark e armazenamento dos DataFrames em um dicionário para facilitar as análises.

In [0]:
base_path = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    "batch-data/"
)

dfs = {}

for tabela in tabelas:

    caminho_arquivo = f"{base_path}{tabela}.csv"

    df = (
        spark.read
            .options(**config)
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(caminho_arquivo)
    )

    dfs[tabela] = df

    print(f"{tabela}: {df.count()} linhas")

#### 7. Validação da leitura

Verificação da quantidade de linhas e colunas carregadas em cada tabela.

In [0]:
for nome, df in dfs.items():
    print(
        f"{nome}: "
        f"{df.count()} linhas | "
        f"{len(df.columns)} colunas"
    )

#### 8. Visualização dos dados

Visualização inicial dos registros para entendimento da estrutura e conteúdo das tabelas.

In [0]:
display(dfs["physical_lojas"])

In [0]:
display(dfs["physical_produtos_pereciveis"])

#### 9. Análise exploratória

Avaliação da estrutura e qualidade dos dados antes da carga no SQL Server.

##### 9.1 Estrutura das tabelas

Verificação das colunas e dos tipos de dados identificados pelo Spark.

In [0]:
for nome, df in dfs.items():
    print(f"\nTABELA: {nome}")
    df.printSchema()

##### 9.2 Volume dos dados

Verificação da quantidade de registros e colunas de cada tabela.

In [0]:
for nome, df in dfs.items():
    print(
        f"{nome}: "
        f"{df.count()} registros | "
        f"{len(df.columns)} colunas"
    )

##### 9.3 Valores nulos

Identificação da quantidade de valores nulos em cada coluna.

In [0]:
from pyspark.sql import functions as F

for nome, df in dfs.items():

    print(f"\nTABELA: {nome}")

    nulos = df.select([
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ])

    display(nulos)

##### 9.4 Registros duplicados

Verificação da existência de registros totalmente duplicados nas tabelas.

In [0]:
for nome, df in dfs.items():

    total = df.count()
    distintos = df.distinct().count()
    duplicados = total - distintos

    print(
        f"{nome}: "
        f"{duplicados} registros duplicados"
    )

##### 9.5 Estatísticas descritivas

Análise das principais estatísticas das colunas numéricas para identificação de valores fora do padrão.

In [0]:
for nome, df in dfs.items():

    print(f"\nTABELA: {nome}")

    display(df.describe())

In [0]:
for nome, df in dfs.items():

    print(f"\n{'=' * 60}")
    print(f"TABELA: {nome}")
    print(f"{'=' * 60}")

    df.printSchema()

    display(df.limit(10))

##### Validação das chaves

Verificação da unicidade dos identificadores das tabelas.

In [0]:
from pyspark.sql import functions as F

# Lojas
display(
    dfs["physical_lojas"]
        .groupBy("id_loja")
        .count()
        .filter(F.col("count") > 1)
)

# Produtos
display(
    dfs["physical_produtos_pereciveis"]
        .groupBy("sku")
        .count()
        .filter(F.col("count") > 1)
)

**Conclusão:** não foram identificadas duplicidades nos campos `id_loja` e `sku`.  
Os campos apresentam valores únicos em suas respectivas tabelas e podem ser considerados chaves candidatas para identificação dos registros.

##### Distribuição das lojas

Análise da distribuição das lojas por estado e dos pesos de venda cadastrados.

In [0]:
display(
    dfs["physical_lojas"]
        .groupBy("estado_loja")
        .count()
        .orderBy(F.desc("count"))
)

In [0]:
display(
    dfs["physical_lojas"]
        .groupBy("peso_vendas")
        .count()
        .orderBy("peso_vendas")
)

In [0]:
display(
    dfs["physical_lojas"]
        .select(
            F.min("peso_vendas").alias("peso_minimo"),
            F.max("peso_vendas").alias("peso_maximo"),
            F.round(F.avg("peso_vendas"), 2).alias("peso_medio")
        )
)

##### Peso de vendas por estado

Análise da distribuição do peso de vendas das lojas entre os estados.


In [0]:
from pyspark.sql import functions as F

peso_por_estado = (
    dfs["physical_lojas"]
        .groupBy("estado_loja")
        .agg(
            F.count("*").alias("qtd_lojas"),
            F.sum("peso_vendas").alias("peso_total"),
            F.round(F.avg("peso_vendas"), 2).alias("peso_medio"),
            F.min("peso_vendas").alias("peso_minimo"),
            F.max("peso_vendas").alias("peso_maximo")
        )
        .orderBy(F.desc("peso_medio"))
)

display(peso_por_estado)

In [0]:
display(
    dfs["physical_lojas"]
        .select(
            "estado_loja",
            "cidade_loja",
            "nome_loja",
            "peso_vendas"
        )
        .orderBy(
            "estado_loja",
            F.desc("peso_vendas")
        )
)

##### Distribuição dos produtos

Análise da distribuição dos produtos por categoria, subcategoria e unidade de medida.

In [0]:
display(
    dfs["physical_produtos_pereciveis"]
        .groupBy("categoria_pai")
        .count()
        .orderBy(F.desc("count"))
)

In [0]:
display(
    dfs["physical_produtos_pereciveis"]
        .groupBy("subcategoria")
        .count()
        .orderBy(F.desc("count"))
)

In [0]:
display(
    dfs["physical_produtos_pereciveis"]
        .groupBy("unidade_medida")
        .count()
        .orderBy(F.desc("count"))
)

##### Análise dos preços

Verificação dos preços cadastrados e identificação de possíveis valores inconsistentes.

In [0]:
display(
    dfs["physical_produtos_pereciveis"]
        .filter(F.col("preco_lista") <= 0)
)

In [0]:
display(
    dfs["physical_produtos_pereciveis"]
        .groupBy("categoria_pai")
        .agg(
            F.count("*").alias("qtd_produtos"),
            F.round(F.avg("preco_lista"), 2).alias("preco_medio"),
            F.round(F.min("preco_lista"), 2).alias("preco_minimo"),
            F.round(F.max("preco_lista"), 2).alias("preco_maximo")
        )
        .orderBy(F.desc("preco_medio"))
)

#### 10. Conexão com o SQL Server

Configuração e validação da conexão com o banco de dados de destino.

In [0]:
jdbc_hostname = "srv-database-intership.database.windows.net"
jdbc_database = "internshipDatabase"
jdbc_port = 1433

jdbc_username = "<SQL_USERNAME>"
jdbc_password = "<SQL_PASSWORD>"

jdbc_url = (
    f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};"
    f"databaseName={jdbc_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

##### Validação da conexão

Teste de acesso ao banco de dados antes da gravação das tabelas.

In [0]:
teste_sql = (
    spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("query", "SELECT 1 AS teste_conexao")
        .option("user", jdbc_username)
        .option("password", jdbc_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
)

display(teste_sql)

In [0]:
schemas = (
    spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option(
            "query",
            """
            SELECT name
            FROM sys.schemas
            WHERE name = 'squad3'
            """
        )
        .option("user", jdbc_username)
        .option("password", jdbc_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
)

display(schemas)

#### 11. Carga no SQL Server

Gravação das tabelas de lojas físicas e produtos perecíveis no schema `squad3`.

In [0]:
(
    dfs["physical_lojas"]
        .write
        .format("sqlserver")
        .option("host", jdbc_hostname)
        .option("port", "1433")
        .option("database", jdbc_database)
        .option("dbtable", "squad3.physical_lojas")
        .option("user", jdbc_username)
        .option("password", jdbc_password)
        .mode("overwrite")
        .save()
)

print("squad3.physical_lojas salva com sucesso!")

##### Gravação no SQL Server

Como o notebook utiliza Serverless Compute, a gravação é realizada utilizando o datasource `sqlserver`, compatível com operações de escrita nesse ambiente.

In [0]:
(
    dfs["physical_produtos_pereciveis"]
        .write
        .format("sqlserver")
        .option("host", jdbc_hostname)
        .option("port", "1433")
        .option("database", jdbc_database)
        .option(
            "dbtable",
            "squad3.physical_produtos_pereciveis"
        )
        .option("user", jdbc_username)
        .option("password", jdbc_password)
        .mode("overwrite")
        .save()
)

print("squad3.physical_produtos_pereciveis salva com sucesso!")

##### Validação da carga

Validação das tabelas gravadas no SQL Server e comparação da quantidade de registros com os dados de origem.

In [0]:
tabelas_sql = [
    "physical_lojas",
    "physical_produtos_pereciveis"
]

for tabela in tabelas_sql:

    df_sql = (
        spark.read
            .format("sqlserver")
            .option("host", jdbc_hostname)
            .option("port", "1433")
            .option("database", jdbc_database)
            .option("dbtable", f"squad3.{tabela}")
            .option("user", jdbc_username)
            .option("password", jdbc_password)
            .load()
    )

    qtd_origem = dfs[tabela].count()
    qtd_sql = df_sql.count()

    print(
        f"{tabela} | "
        f"Origem: {qtd_origem} | "
        f"SQL Server: {qtd_sql}"
    )